In [173]:
# import scvi
import re
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os
from collections import Counter
import matplotlib.pyplot as plt
# from scvi.model.utils import mde
from random import sample
from random import seed
from bokeh.palettes import Turbo256
# from concurrent.futures import ProcessPoolExecutor
import pandas as pd
import polars as pl
import glob
import itertools
from math import floor
import scrublet as scr # doublet detection
import contextlib


plt.rcParams["figure.figsize"] = (4,3)
os.chdir("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison")
sc.set_figure_params(figsize=(4, 4))

# for white background of figures (only for docs rendering)
%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

In [174]:
from scanpy.readwrite import is_valid_filename,read
from scanpy._utils import Empty, _empty
from pathlib import Path
def _read_v3_10x_mtx(
    path,
    var_names='gene_symbols',
    make_unique=True,
    cache=False,
    cache_compression=_empty,
    *,
    prefix=""):
    """
    Read mtx from output from Cell Ranger v3 or later versions
    """
    path = Path(path)
    adata = read(
        path / f'{prefix}matrix.mtx.gz',
        cache=cache,
        cache_compression=cache_compression,
    ).T  # transpose the data
    genes = pd.read_csv(path / f'{prefix}features.tsv.gz', header=None, sep='\t')
    if var_names == 'gene_symbols':
        var_names = genes[1].values
        if make_unique:
            var_names = anndata.utils.make_index_unique(pd.Index(var_names))
        adata.var_names = var_names
        adata.var['gene_ids'] = genes[0].values
    elif var_names == 'gene_ids':
        adata.var_names = genes[0].values
        adata.var['gene_symbols'] = genes[1].values
    elif var_names == "BGI":
        adata.var_names = genes.iloc[:,0].to_list()
        adata.var['gene_symbols'] = genes.iloc[:,0].to_list()
    else:
        raise ValueError("`var_names` needs to be 'gene_symbols' or 'gene_ids'")
    if var_names != "BGI":
        adata.var['feature_types'] = genes[2].values
    adata.obs_names = pd.read_csv(path / f'{prefix}barcodes.tsv.gz', header=None)[
        0
    ].values
    return adata


In [175]:
sc.read_10x_mtx = _read_v3_10x_mtx

In [176]:
def read_lib(path: str, region_name: str, specie: str, seq_method: str="C4", donor: str="unknown", skip_mtx: bool=False, ad=None):
    if skip_mtx:
        lib = ad
    else:
        lib = sc.read_10x_mtx(path, var_names="BGI", cache_compression="gzip")
    lib.obs["nCount_RNA"] = lib.X.sum(axis=1)
    lib.obs["nFeature_RNA"] = lib.X.astype(bool).sum(axis=1)
    lib.obs["specie"] = specie
    lib.obs["region_name"] = region_name
    lib.obs["batch"] = os.path.basename(path)
    lib.obs["donor"] = donor
    lib.obs["seq_method"] = seq_method
    lib.obs["cell_type"] = None
    
    
    ### pre-filtering
    sc.pp.filter_cells(lib, min_counts=800)
    
    # doublet detection
#     print(lib.obs.batch[0])
#     print(lib.shape)
#     with contextlib.redirect_stdout(None):
#         scrub = scr.Scrublet(lib.X, expected_doublet_rate=0.12, random_state=42)
#         doublet_scores, predicted_doublets = scrub.scrub_doublets(n_prin_comps=30)
#     print(predicted_doublets)

#     if predicted_doublets is None:
#         pass
#     else:
#         lib = lib[~predicted_doublets]
#     print(lib.shape)
    
    print(lib.shape)
    sc.pp.scrublet(lib, expected_doublet_rate=0.15, verbose=True)
    lib = lib[~lib.obs.predicted_doublet]
    print(lib.shape)
    return(lib)

def read_lib_l(pathl: list[str], region_name: str, specie: str, 
               prefix: str = "", postfix: str = "", seq_method: str="C4", donor: str="unknown"):
    return([read_lib(os.path.join(prefix, p + postfix), region_name, specie, seq_method, donor) for p in pathl])


def read_lib_l2(pathl: str, region_name: str, specie: str, 
               prefix: str = "", seq_method: str="C4", donor: str="unknown"):
    pathl = [glob.glob(os.path.join(prefix, p) + "-*") for p in pathl.split(";")]
    pathl = list(itertools.chain.from_iterable(pathl))
    return(read_lib_l(pathl, region_name, specie, prefix, "", seq_method, donor))

In [177]:
def filter_ad(adata, inplace=False, nc_cutoff=800, cfr_cutoff=1.2):
    adata.obs["cf_ratio"] = adata.obs.nCount_RNA/adata.obs.nFeature_RNA
    if inplace is True:
        adata = adata[(adata.obs.nFeature_RNA > nc_cutoff) & (adata.obs.cf_ratio > cfr_cutoff)]
        return(adata)
    else:
        print(adata[(adata.obs.nFeature_RNA > nc_cutoff) & (adata.obs.cf_ratio > cfr_cutoff)].obs.value_counts("region_name"))

def subset_ad(adata, cell_num, batch_name, platform_num=1):
    """Sample num randomly from batch_name from the provided clusters

    Parameters
    ----------
    adata
        AnnData object
    cell_num
        Number of samples to take from each cluster
    batch_name
        Cluster column name
    platform_num
        Number of data sequencing platform, cell numbers will be divided equally to each platform  
    
    Returns
    -------
    AnnData
        Annotated data matrix with sampled cells from the clusters
    """
    
    rng = np.random.default_rng(42)
    l = []
    
    ##### tackle different sequencing platform
    cell_num = floor(cell_num / platform_num)
    
    for k, v in adata.obs.groupby(batch_name).indices.items():
        if len(v) < cell_num:
            l.append(v)
        else:
            l.append(rng.choice(v, cell_num, replace=False)) 
    return(adata[np.concatenate(l)])

### load polars table named conv_tb

In [178]:
# pl.read_csv("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison/conversion_table.ma.20241016.tsv", 
#                       separator='\t').with_columns(
#     pl.when((pl.col("humanGene") == pl.col("marmosetGene")) & (pl.col("mouseGene").str.to_uppercase() == pl.col("marmosetGene")) & pl.col("macaqueGene").is_null())
#      .then(pl.col("humanGene"))
#      .otherwise(pl.col("macaqueGene"))
#      .alias("macaqueGene"),
#     pl.when((pl.col("humanGene") == pl.col("macaqueGene")) & (pl.col("mouseGene").str.to_uppercase() == pl.col("macaqueGene")) & pl.col("marmosetGene").is_null())
#      .then(pl.col("humanGene"))
#      .otherwise(pl.col("marmosetGene"))
#      .alias("marmosetGene")
#         ).unique(
#             subset=["humanID", "mouseID","marmosetID", "marmosetID"]
#         ).select(["humanID", "mouseID", "macaqueGene", "marmosetGene", "humanGene"]).filter(
#             pl.col("mouseID").is_in(conv_tb["mouseID"].value_counts().filter(pl.col("count") != 1).get_column("mouseID").to_list()).not_() &
#             pl.col("humanID").is_in(conv_tb["humanID"].value_counts().filter(pl.col("count") != 1).get_column("humanID").to_list()).not_() &
#             pl.col("macaqueGene").is_in(conv_tb["macaqueGene"].value_counts().filter(pl.col("count") != 1).get_column("macaqueGene").to_list()).not_() &
#             pl.col("marmosetGene").is_in(conv_tb["marmosetGene"].value_counts().filter(pl.col("count") != 1).get_column("marmosetGene").to_list()).not_() &
#             pl.col("humanGene").is_in(conv_tb["humanGene"].value_counts().filter(pl.col("count") != 1).get_column("humanGene").to_list()).not_()
# ).drop_nulls().filter(pl.col("humanGene") ==  "SLC17A8")

In [179]:
conv_tb = pl.read_csv("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison/conversion_table.ma.20241016.tsv", 
                      separator='\t').with_columns(
    pl.when((pl.col("humanGene") == pl.col("marmosetGene")) & (pl.col("mouseGene").str.to_uppercase() == pl.col("marmosetGene")) & pl.col("macaqueGene").is_null())
     .then(pl.col("humanGene"))
     .otherwise(pl.col("macaqueGene"))
     .alias("macaqueGene"),
    pl.when((pl.col("humanGene") == pl.col("macaqueGene")) & (pl.col("mouseGene").str.to_uppercase() == pl.col("macaqueGene")) & pl.col("marmosetGene").is_null())
     .then(pl.col("humanGene"))
     .otherwise(pl.col("marmosetGene"))
     .alias("marmosetGene")
        ).unique(
            subset=["humanID", "mouseID","marmosetID", "marmosetID"]
        )
conv_tb = conv_tb.select(["humanID", "mouseID", "macaqueGene", "marmosetGene", "humanGene"]).filter(
            pl.col("mouseID").is_in(conv_tb["mouseID"].value_counts().filter(pl.col("count") != 1).get_column("mouseID").to_list()).not_() &
            pl.col("humanID").is_in(conv_tb["humanID"].value_counts().filter(pl.col("count") != 1).get_column("humanID").to_list()).not_() &
            pl.col("macaqueGene").is_in(conv_tb["macaqueGene"].value_counts().filter(pl.col("count") != 1).get_column("macaqueGene").to_list()).not_() &
            pl.col("marmosetGene").is_in(conv_tb["marmosetGene"].value_counts().filter(pl.col("count") != 1).get_column("marmosetGene").to_list()).not_() &
            pl.col("humanGene").is_in(conv_tb["humanGene"].value_counts().filter(pl.col("count") != 1).get_column("humanGene").to_list()).not_()
).drop_nulls()
conv_tb.write_csv("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison/conversion_table.ma.20241016.dedup.csv")
# conv_tb = pl.read_csv("/cluster/home/majianyun/data/Hypothalamus/conversion_table.liu.20221031.tsv", separator='\t')
# conv_tb = pl.read_csv("/cluster/home/majianyun/data/Hypothalamus/conv_tb.gprofiler.20230729.csv")
print(conv_tb.head(50))
print(conv_tb.shape)

shape: (50, 5)
┌─────────────────┬────────────────────┬─────────────┬──────────────┬───────────┐
│ humanID         ┆ mouseID            ┆ macaqueGene ┆ marmosetGene ┆ humanGene │
│ ---             ┆ ---                ┆ ---         ┆ ---          ┆ ---       │
│ str             ┆ str                ┆ str         ┆ str          ┆ str       │
╞═════════════════╪════════════════════╪═════════════╪══════════════╪═══════════╡
│ ENSG00000184895 ┆ ENSMUSG00000069036 ┆ SRY         ┆ SRY          ┆ SRY       │
│ ENSG00000154639 ┆ ENSMUSG00000044860 ┆ CXADR       ┆ CXADR        ┆ CXADR     │
│ ENSG00000154721 ┆ ENSMUSG00000053062 ┆ JAM2        ┆ JAM2         ┆ JAM2      │
│ ENSG00000243927 ┆ ENSMUSG00000039680 ┆ MRPS6       ┆ MRPS6        ┆ MRPS6     │
│ …               ┆ …                  ┆ …           ┆ …            ┆ …         │
│ ENSG00000160199 ┆ ENSMUSG00000006705 ┆ PKNOX1      ┆ PKNOX1       ┆ PKNOX1    │
│ ENSG00000141425 ┆ ENSMUSG00000040446 ┆ RPRD1A      ┆ RPRD1A       ┆ RPRD1A    │
│

### define used region

In [180]:
regions = ["Hypothalamus", "Hippocampus", "Cerebellum", "Visual", "PFC", "Motor"]
              # , "Clastrum", "Insula"

#### marmoset & macaque

In [58]:
mar_mac_region = pd.read_csv("/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_2022_10_10/SampleInfoAll.txt", sep='\t')
mac_region2 = mar_mac_region.loc[mar_mac_region.Name1.str.startswith("MQ")]
mar_region2 = mar_mac_region.loc[mar_mac_region.Name1.str.startswith("MT")]
mac_region2.head()

,Name1,Name2,BrainArea,Region,SampleName
0,MQC286R-1,猴大脑,"29,30",Cortex,6434-2-220817
1,MQC286R-1,猴大脑,"29,30",Cortex,6434-1-220817
2,MQC286R-1/2,猴大脑,1/2,Cortex,6086-2-220805
3,MQC286R-1/2,猴大脑,1/2,Cortex,6086-1-220805
4,MQC286R-10mc,猴大脑,10mc,Cortex,6087-2-220804


In [59]:
mar_region = pd.read_csv("/cluster/share/hypo_group/snRNA_marmoset_brain/Sample_names.csv")
mar_region.head()

,Number,Library,number,slice,region,class
0,M012L-4,9853-1-220316,4,T413-T422,"A14R,A13a,A13b,A13M,A13L,A47M(12M),A47L(12L),A...",皮层
1,M012L-4,9853-2-220316,4,T413-T422,"A14R,A13a,A13b,A13M,A13L,A47M(12M),A47L(12L),A...",皮层
2,M012L-4,9853-3-220316,4,T413-T422,"A14R,A13a,A13b,A13M,A13L,A47M(12M),A47L(12L),A...",皮层
3,M012L-5,9854-1-220316,5,T413-T422,白质和碎渣,NaN
4,M012L-5,9854-2-220316,5,T413-T422,白质和碎渣,NaN


In [60]:
mar_fgp = sc.concat([sc.read_h5ad(i) for i in glob.glob("/cluster/share/hypo_group/snRNA_marmoset_brain/marmoset_FGP/*.h5ad")], join="outer")

mar_fgp = mar_fgp[mar_fgp.obs.sex == "male"]

mar_fgp = mar_fgp[~mar_fgp.obs_names.duplicated()]

mar_fgp_raw = mar_fgp.raw.to_adata()

mar_fgp = mar_fgp_raw

mar_fgp.obs["seq_method"] = "10X"
mar_fgp.obs["specie"] = "marmoset"
mar_fgp.obs["donor"] = mar_fgp.obs.donor_id
mar_fgp.obs["batch"] = mar_fgp.obs.seq_pool
mar_fgp.obs["nCount_RNA"] = mar_fgp.X.sum(axis=1)
mar_fgp.obs["nFeature_RNA"] = mar_fgp.X.astype(bool).sum(axis=1)

/cluster/home/majianyun/micromamba/envs/scvi/lib/python3.10/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/cluster/home/majianyun/micromamba/envs/scvi/lib/python3.10/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [62]:
fgp_dict = {i:(j if j is not None else i+' no_name') for i,j in zip(conv_tb.get_column("humanID"), conv_tb.get_column("marmosetGene"))} # no None

In [63]:
mar_fgp_ori = mar_fgp
mar_fgp = mar_fgp[:,[i in conv_tb.get_column("humanID") for i in mar_fgp.var_names]]
mar_fgp.var_names = [fgp_dict[i] for i in mar_fgp.var_names]

In [64]:
mar_fgp_hippo = mar_fgp[mar_fgp.obs.region == "hipp"]
mar_fgp_hippo.obs["region_name"] = "Hippocampus"
mar_fgp_cb = mar_fgp[mar_fgp.obs.region == "cbm"]
mar_fgp_cb.obs["region_name"] = "Cerebellum"
mar_fgp_hy = mar_fgp[mar_fgp.obs.region == "hyp"]
mar_fgp_hy.obs["region_name"] = "Hypothalamus"
mar_fgp_mot = mar_fgp[mar_fgp.obs.region == "m1"]
mar_fgp_mot.obs["region_name"] = "Motor"
mar_fgp_vis = mar_fgp[(mar_fgp.obs.region == "v1") | (mar_fgp.obs.region == "v2")]
mar_fgp_vis.obs["region_name"] = "Visual"
mar_fgp_pfc = mar_fgp[(mar_fgp.obs.region == "pfc") & (mar_fgp.obs.anatomical_name != "orbitofrontal cortex")]
mar_fgp_pfc.obs["region_name"] = "PFC"

/tmp/ipykernel_43578/104829113.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mar_fgp_hippo.obs["region_name"] = "Hippocampus"
/tmp/ipykernel_43578/104829113.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mar_fgp_cb.obs["region_name"] = "Cerebellum"
/tmp/ipykernel_43578/104829113.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mar_fgp_hy.obs["region_name"] = "Hypothalamus"
/tmp/ipykernel_43578/104829113.py:8: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mar_fgp_mot.obs["region_name"] = "Motor"
/tmp/ipykernel_43578/104829113.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mar_fgp_vis.obs["region_name"] = "Visual"
/tmp/ipykernel_43578/104829113.py:12: ImplicitModificationWarning: Trying t

In [ ]:
mar_hippo = read_lib_l2("9890;9940", 
           "Hippocampus", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain", donor="MT12") + [mar_fgp_hippo]
mar_hippo

(25119, 31520)
(21944, 31520)


In [ ]:
mar_cb = read_lib_l2("9918;9922;9925;9928;9932;9946;9944", 
           "Cerebellum", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain", donor="MT12") + [mar_fgp_cb]
        #+ read_lib_l2("6690;6689;6691;6687;6688;6692;6695", 
        #"Cerebellum", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain/marmoset2_cerebellum_snRNA", donor="MT27")
mar_cb

In [ ]:
mar_vis = read_lib_l2("9923;9926;9930;9947;9945;9924;9927;9931;9921;9917;9913;9912;9897", 
           "Visual", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain", donor="MT12") + [mar_fgp_vis]
mar_vis

In [ ]:
mar_pfc = read_lib_l2("9853;9867;9865;9870", 
           "PFC", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain", donor="MT12") + [mar_fgp_pfc]
mar_pfc

In [ ]:
mar_mot = read_lib_l2("9877;9866",
           "Motor", "marmoset", "/cluster/share/hypo_group/snRNA_marmoset_brain", donor="MT12") + [mar_fgp_mot]
mar_mot

In [ ]:
mar_hy = sc.read_h5ad("../snRNA_data_for_transfer/marmoset.h5ad")
mar_hy.obs["region_name"] = "Hypothalamus"
mar_hy.obs["seq_method"] = "C4"
mar_hy
mar_hy_map = pd.read_csv("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison/marmoset.donor_lib.csv")
mar_hy_map = {i:j for i,j in zip(mar_hy_map["library_minimal"], mar_hy_map["donor_id"])}
mar_hy_map
mar_hy.obs["batch"] = [s[:-6] for s in mar_hy.obs.batch]
mar_hy.obs["donor"] = [mar_hy_map[l] for l in mar_hy.obs["batch"].to_list()]

In [ ]:
mar = sc.concat(mar_hippo + mar_cb + #mar_bs +
                mar_vis + mar_pfc + mar_mot + 
                [mar_hy] + [mar_fgp_hy], join="outer")

#### macaque

In [ ]:
mac_region = pd.read_csv("/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14/sampleInfo.txt", sep='\t')
mac_region.head()

In [ ]:
mac_name = pd.read_csv("/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14/sampleName.txt", sep='\t', header=None, names=["lib"])
mac_name["sample"] = [str(i[0]) for i in mac_name["lib"].str.split('-', n=1)]  #### mac_mar_name["sample"] = 
mac_region = pd.merge(mac_name, mac_region, on="sample", how="left")
mac_region.head()

In [ ]:
mac_hippo = read_lib_l2("3T26;13T42;11T35;53T37;64T49", 
           "Hippocampus", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14", donor="MQ179"
            ) + read_lib_l2("7238;7239;7241;7242", 
           "Hippocampus", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_202302_23", donor="MQ279R"
            ) + read_lib_l2("7250;7255;7249;7252;7253;7254;7251", 
           "Hippocampus", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_202302_23", donor="MQC286R")
mac_hippo

In [ ]:
mac_cb = read_lib_l2("24T58;41T92", 
           "Cerebellum", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14/", donor="MQ179"
            ) + read_lib_l2("6703;6698;6710;6705;6716;6718;6721;6709;6707;6706;6702;6722;6704;6720;6723;6713;6711;6699;6700;6701", 
            "Cerebellum", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_2022_10_10", donor="MQC286R")
mac_cb

In [ ]:
mac_vis= read_lib_l2("V1T58;27T74;38T74;42T92;43T92;36T74;V2T58;28T74;37T74;44T92;45T92;14T42;29T74;39T74;40T74;23T58;26T74;22T58;34T74", 
           "Visual", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14", donor="MQ179"
            ) + read_lib_l2("6466;6467;6472;6471;6476;6477;6479;6470;6473;6480;6478;6469", 
            "Visual", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_2022_10_10", donor="MQC286R")
mac_vis

In [ ]:
mac_pfc = read_lib_l2("116T143;118;117T143;108T113;107T113;111T127;115T143;83T73;100T99;106T113;93T85;112T127",
           "PFC", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14", donor="MQ179"
            ) + read_lib_l2("116T143;118;117T143;108T113;107T113;111T127;115T143;83T73;100T99;106T113;93T85;112T127",
           "PFC", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_2022_10_10", donor="MQC286R")
mac_pfc

In [ ]:
mac_mot = read_lib_l2("1T35;46T37;57T49;69T61;82T73", 
           "Motor", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V1_snRNA_data/data_2021_08_14/", donor="MQ179"
            ) + read_lib_l(mac_region2.loc[mac_region2['BrainArea'] == "F1"].SampleName.to_list(),
            "Motor", "macaque", "/cluster/share/hypo_group/snRNA_macaque_brain/ma_fas_V2_snRNA_data/data_2022_10_10", donor="MQC286R")

mac_mot

In [ ]:
mac_hy = sc.read_h5ad("../snRNA_data_for_transfer/macaque.h5ad")
mac_hy.obs["region_name"] = "Hypothalamus"
mac_hy.obs["seq_method"] = "C4"
mac_hy

mac_hy.obs.batch = [i[3] if j == "MJ" else j[:-6] for i,j in zip(mac_hy.obs_names.str.split('_'), mac_hy.obs["batch"])]

mac_hy_map = pd.read_csv("/cluster/home/majianyun/data/Hypothalamus/scvi/region_comparison/macaque.donor_lib.csv")
mac_hy_map = {i:j for i,j in zip(mac_hy_map["library_minimal"], mac_hy_map["donor_id"])}
mac_hy_map

In [ ]:
mac_hy.obs["donor"] = [mac_hy_map[l] for l in mac_hy.obs["batch"].to_list()]

In [ ]:
mac_hy.obs.donor.value_counts()

In [ ]:
mac = sc.concat(mac_hippo + mac_cb + mac_mot + 
                mac_vis + mac_pfc + #mac_tm + 
                [mac_hy], join="outer")

#### mouse

In [182]:
mou_meta = pd.read_csv("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/cell_metadata.csv")
mou_meta.head()

,cell_label,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,donor_sex,dataset_label,x,y,cluster_alias
0,GCGAGAAGTTAAGGGC-410_B05,GCGAGAAGTTAAGGGC,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.146826,-3.086639,1
1,AATGGCTCAGCTCCTT-411_B06,AATGGCTCAGCTCCTT,411_B06,L8TX_201029_01_E10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550851,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.138481,-3.022000,1
2,AACACACGTTGCTTGA-410_B05,AACACACGTTGCTTGA,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.472557,-2.992709,1
3,CACAGATAGAGGCGGA-410_A05,CACAGATAGAGGCGGA,410_A05,L8TX_201029_01_A10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.379622,-3.043442,1
4,AAAGTGAAGCATTTCG-410_B05,AAAGTGAAGCATTTCG,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.909480,-2.601536,1


In [183]:
mou_ctx1 = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/WMB-10Xv3-Isocortex-1-raw.h5ad")
mou_ctx1.obs.drop(columns=['cell_barcode', 'library_label'], inplace=True)
mou_ctx1.obs = mou_ctx1.obs.join(mou_meta.set_index("cell_label"))
mou_ctx1.obs["batch"] = "mou_ctx1"
mou_ctx1.obs.region_of_interest_acronym.value_counts()

region_of_interest_acronym
MOp                 76033
RSP                 23050
ACA                 21550
AI                  20427
PL-ILA-ORB          17164
VIS                 14681
SS-GU-VISC          14213
AUD-TEa-PERI-ECT    14206
MO-FRP              14172
VIS-PTLp            12174
Name: count, dtype: int64

In [184]:
mou_ctx2 = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/WMB-10Xv3-Isocortex-2-raw.h5ad")
mou_ctx2.obs.drop(columns=['cell_barcode', 'library_label'], inplace=True)
mou_ctx2.obs = mou_ctx2.obs.join(mou_meta.set_index("cell_label"))
mou_ctx2.obs["batch"] = "mou_ctx2"
mou_ctx2.obs.region_of_interest_acronym.value_counts()

region_of_interest_acronym
MOp                 48468
AI                  25773
ACA                 23946
SS-GU-VISC          23098
MO-FRP              22494
RSP                 21605
PL-ILA-ORB          17899
VIS                 16201
VIS-PTLp            15339
AUD-TEa-PERI-ECT    12714
Name: count, dtype: int64

In [185]:
mou_ctx = sc.concat([mou_ctx1, mou_ctx2],  join="outer")

In [186]:
mou_ctx = mou_ctx[mou_ctx.obs.donor_sex == "M"]

In [187]:
mou_ctx.obs.region_of_interest_acronym.value_counts()

region_of_interest_acronym
MOp                 48683
ACA                 31831
AI                  30631
RSP                 26728
PL-ILA-ORB          23938
MO-FRP              21717
SS-GU-VISC          21266
VIS                 14446
AUD-TEa-PERI-ECT    11504
VIS-PTLp            11441
Name: count, dtype: int64

In [188]:
mou_ctx.obs["specie"] = "mouse"
mou_ctx.obs["seq_method"] = "10X"

/tmp/ipykernel_147401/2577890686.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_ctx.obs["specie"] = "mouse"


In [189]:
mou_ctx.obs["batch"] = mou_ctx.obs.library_label
mou_ctx.obs["donor"] = mou_ctx.obs.donor_label
mou_ctx.obs["nCount_RNA"] = mou_ctx.X.sum(axis=1)
mou_ctx.obs["nFeature_RNA"] = mou_ctx.X.astype(bool).sum(axis=1)

In [190]:
mou_pfc = mou_ctx[mou_ctx.obs.region_of_interest_acronym == "PL-ILA-ORB"]
mou_pfc.obs["region_name"] = "PFC"
mou_vis = mou_ctx[(mou_ctx.obs.region_of_interest_acronym == "VIS") | (mou_ctx.obs.region_of_interest_acronym == "VIS-PTLp")]
mou_vis.obs["region_name"] = "Visual"
mou_motor = mou_ctx[mou_ctx.obs.region_of_interest_acronym == "MOp"]
mou_motor.obs["region_name"] = "Motor"

/tmp/ipykernel_147401/2360917058.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_pfc.obs["region_name"] = "PFC"
/tmp/ipykernel_147401/2360917058.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_vis.obs["region_name"] = "Visual"
/tmp/ipykernel_147401/2360917058.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_motor.obs["region_name"] = "Motor"


In [191]:
mou_hippo = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/WMB-10Xv3-HPF-raw.h5ad")
mou_hippo.obs.drop(columns=['cell_barcode', 'library_label'], inplace=True)
mou_hippo.obs = mou_hippo.obs.join(mou_meta.set_index("cell_label"))
mou_hippo = mou_hippo[mou_hippo.obs.donor_sex == "M"]
mou_hippo.obs["specie"] = "mouse"
mou_hippo.obs["region_name"] = "Hippocampus"
mou_hippo.obs["seq_method"] = "10X"
mou_hippo.obs["batch"] = mou_hippo.obs.library_label
mou_hippo.obs["donor"] = mou_hippo.obs.donor_label
mou_hippo.obs["nCount_RNA"] = mou_hippo.X.sum(axis=1)
mou_hippo.obs["nFeature_RNA"] = mou_hippo.X.astype(bool).sum(axis=1)

mou_hippo

/tmp/ipykernel_147401/2086182567.py:5: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_hippo.obs["specie"] = "mouse"


AnnData object with n_obs × n_vars = 94928 × 32285
    obs: 'anatomical_division_label', 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'specie', 'region_name', 'seq_method', 'batch', 'donor', 'nCount_RNA', 'nFeature_RNA'
    var: 'gene_symbol'
    uns: 'normalization', 'parent', 'parent_layer', 'parent_rows'

In [192]:
mou_cb = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/WMB-10Xv3-CB-raw.h5ad")
mou_cb.obs.drop(columns=['cell_barcode', 'library_label'], inplace=True)
mou_cb.obs = mou_cb.obs.join(mou_meta.set_index("cell_label"))
mou_cb = mou_cb[mou_cb.obs.donor_sex == "M"]
mou_cb.obs["specie"] = "mouse"
mou_cb.obs["region_name"] = "Cerebellum"
mou_cb.obs["seq_method"] = "10X"
mou_cb.obs["batch"] = mou_cb.obs.library_label
mou_cb.obs["donor"] = mou_cb.obs.donor_label
mou_cb.obs["nCount_RNA"] = mou_cb.X.sum(axis=1)
mou_cb.obs["nFeature_RNA"] = mou_cb.X.astype(bool).sum(axis=1)

mou_cb

/tmp/ipykernel_147401/893343501.py:5: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_cb.obs["specie"] = "mouse"


AnnData object with n_obs × n_vars = 88135 × 32285
    obs: 'anatomical_division_label', 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'specie', 'region_name', 'seq_method', 'batch', 'donor', 'nCount_RNA', 'nFeature_RNA'
    var: 'gene_symbol'
    uns: 'normalization', 'parent', 'parent_layer', 'parent_rows'

In [204]:
mou_hy = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/A_high-resolution_transcriptomic_and_spatial_atlas_of_cell_types_in_the_whole_mouse_brain/TXv3/WMB-10Xv3-HY-raw.h5ad")
mou_hy.obs.drop(columns=['cell_barcode', 'library_label'], inplace=True)
mou_hy.obs = mou_hy.obs.join(mou_meta.set_index("cell_label"))
mou_hy = mou_hy[mou_hy.obs.donor_sex == "M"]
mou_hy.obs["specie"] = "mouse"
mou_hy.obs["region_name"] = "Hypothalamus"
mou_hy.obs["seq_method"] = "10X"
mou_hy.obs["batch"] = mou_hy.obs.library_label
mou_hy.obs["donor"] = mou_hy.obs.donor_label
mou_hy.obs["nCount_RNA"] = mou_hy.X.sum(axis=1)
mou_hy.obs["nFeature_RNA"] = mou_hy.X.astype(bool).sum(axis=1)

mou_hy

/tmp/ipykernel_147401/3253485521.py:5: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  mou_hy.obs["specie"] = "mouse"


AnnData object with n_obs × n_vars = 85837 × 32285
    obs: 'anatomical_division_label', 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'specie', 'region_name', 'seq_method', 'batch', 'donor', 'nCount_RNA', 'nFeature_RNA'
    var: 'gene_symbol'
    uns: 'normalization', 'parent', 'parent_layer', 'parent_rows'

#### human 

In [102]:
hneu = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/Transcriptomic_diversity_of_cell_types_across_the_adult_human_brain/neuron.h5ad")
hneu
hnneu = sc.read_h5ad("/cluster/home/majianyun/data/data_from_paper/Transcriptomic_diversity_of_cell_types_across_the_adult_human_brain/nonneuron.h5ad")
hnneu

AnnData object with n_obs × n_vars = 888263 × 59357
    obs: 'ROIGroup', 'ROIGroupCoarse', 'ROIGroupFine', 'roi', 'organism_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'sex_ontology_term_id', 'development_stage_ontology_term_id', 'donor_id', 'suspension_type', 'dissection', 'fraction_mitochondrial', 'fraction_unspliced', 'cell_cycle_score', 'total_genes', 'total_UMIs', 'sample_id', 'supercluster_term', 'cluster_id', 'subcluster_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'Biotype', 'Chromosome', 'End', 'Gene', 'Start', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'batch_condition', 'schema_version', 'title'
    obsm: 'X_UMAP', 'X_tSNE'

In [103]:
hdad = sc.concat([hneu, hnneu], join="outer")

In [121]:
hdad = hdad[hdad.obs.sex == "male"]
hdad.obs["specie"] = "human"
hdad.obs["batch"] = hdad.obs.donor_id
hdad.obs["donor"] = hdad.obs.donor_id
hdad.obs["seq_method"] = "10X"
hdad.obs["nCount_RNA"] = hdad.X.sum(axis=1)
hdad.obs["nFeature_RNA"] = hdad.X.astype(bool).sum(axis=1)

/tmp/ipykernel_147401/391737928.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hdad.obs["specie"] = "human"


In [122]:
hum_motor = hdad[hdad.obs.dissection == "Cerebral cortex (Cx) - Precentral gyrus (PrCG) - Primary motor cortex - M1C"]
hum_motor.obs["region_name"] = "Motor"
hum_pfc = hdad[hdad.obs.dissection == "Cerebral cortex (Cx) - Inferior frontal gyrus (IFG) - Ventrolateral prefrontal cortex - A44-A45"]
hum_pfc.obs["region_name"] = "PFC"
hum_vis = hdad[hdad.obs.dissection == "Cerebral cortex (Cx) - Lingual gyrus (LiG) - Primary Visual Cortex - V1C"]
hum_vis.obs["region_name"] = "Visual"
hum_hippo =  hdad[hdad.obs.tissue == "hippocampal formation"]
hum_hippo.obs["region_name"] = "Hippocampus"
hum_cb =  hdad[hdad.obs.tissue == "cerebellum"]
hum_cb.obs["region_name"] = "Cerebellum"
hum_hy =  hdad[hdad.obs.tissue == "hypothalamus"]
hum_hy.obs["region_name"] = "Hypothalamus"

/tmp/ipykernel_147401/2795937067.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hum_motor.obs["region_name"] = "Motor"
/tmp/ipykernel_147401/2795937067.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hum_pfc.obs["region_name"] = "PFC"
/tmp/ipykernel_147401/2795937067.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hum_vis.obs["region_name"] = "Visual"
/tmp/ipykernel_147401/2795937067.py:8: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hum_hippo.obs["region_name"] = "Hippocampus"
/tmp/ipykernel_147401/2795937067.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  hum_cb.obs["region_name"] = "Cerebellum"
/tmp/ipykernel_147401/2795937067.py:12: ImplicitModificationWarning: Trying to modify attri

In [123]:
hum = sc.concat([hum_motor, hum_pfc, hum_vis, hum_hippo, hum_cb, hum_hy], join="outer")

#### integration

In [212]:
def integration_preprocess(mou, mac, mar, hum, conv_tb, downsample=20000, filter_cells=True):
    if filter_cells:
        moud = subset_ad(filter_ad(mou, inplace=True), downsample, ["region_name", "seq_method"], mou.obs.seq_method.nunique())
        moud.obs.index.name = "index"
        macd = subset_ad(filter_ad(mac, inplace=True), downsample, ["region_name", "seq_method"], mac.obs.seq_method.nunique())
        macd.obs.index.name = "index"
        mard = subset_ad(filter_ad(mar, inplace=True), downsample, ["region_name", "seq_method"], mar.obs.seq_method.nunique())
        mard.obs.index.name = "index"
        humd = subset_ad(filter_ad(hum, inplace=True), downsample, ["region_name", "seq_method"], hum.obs.seq_method.nunique())
        humd.obs.index.name = "index"
    else:
        moud = mou
        mard = mar
        macd = mac
        humd = hum
        
    ### mouse
    moud.var["ori_name"] = moud.var_names
    moud = moud[:, [i in conv_tb.get_column("mouseID").to_list() for i in moud.var.index.to_list()]]
    moud.obs.index.name = "index"
    mouse_dict = {}
    for i in moud.var_names:
        mouse_dict[i] = conv_tb.filter(pl.col("mouseID") == i).get_column("humanGene")[0]
    moud.var_names = [mouse_dict[i] for i in moud.var_names]
    moud.var
        
    ### marmoset
    mard.var["ori_name"] = mard.var_names
    mard = mard[:, [i in conv_tb.get_column("marmosetGene").to_list() for i in mard.var.index.to_list()]]
    mard.obs.index.name = "index"
    marmoset_dict = {}
    for i in mard.var_names:
        marmoset_dict[i] = conv_tb.filter(pl.col("marmosetGene") == i).get_column("humanGene")[0]
    mard.var_names = [marmoset_dict[i] for i in mard.var_names]
    mard.var
    
    ### macaque
    macd.var["ori_name"] = macd.var_names
    macd = macd[:, [i in conv_tb.get_column("macaqueGene").to_list() for i in macd.var.index.to_list()]]
    macd.obs.index.name = "index"
    macaque_dict = {}
    for i in macd.var_names:
        macaque_dict[i] = conv_tb.filter(pl.col("macaqueGene") == i).get_column("humanGene")[0]
    macd.var_names = [macaque_dict[i] for i in macd.var_names]
    macd.var
    
    ### human 
    humd.var["ori_name"] = humd.var_names
    humd = humd[:, [i in conv_tb.get_column("humanID").to_list() for i in humd.var.index.to_list()]]
    humd.obs.index.name = "index"
    human_dict = {}
    for i in humd.var_names:
        human_dict[i] = conv_tb.filter(pl.col("humanID") == i).get_column("humanGene")[0]
    humd.var_names = [human_dict[i] for i in humd.var_names]
    humd.var
    
    
    moud.obs_names_make_unique() 
    macd.obs_names_make_unique() 
    mard.obs_names_make_unique()
    humd.obs_names_make_unique()
    
    return(moud, macd, mard, humd)

In [213]:
moud,macd,mard,humd = integration_preprocess(mou, mac, mar, hum, conv_tb)

/tmp/ipykernel_43578/463653476.py:18: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  moud.var["ori_name"] = moud.var_names
/tmp/ipykernel_43578/463653476.py:28: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  mard.var["ori_name"] = mard.var_names
/cluster/home/majianyun/micromamba/envs/scvi/lib/python3.10/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/cluster/home/majianyun/micromamba/envs/scvi/lib/python3.10/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_43578/463653476.py:38: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  macd.var["ori_name"

In [230]:
mard.obs.drop(columns="is_primary_data", inplace=True)
macd.obs.predicted_doublet = macd.obs.predicted_doublet.astype("str")
mard.obs.predicted_doublet = mard.obs.predicted_doublet.astype("str")

In [231]:
macd.write_h5ad("macaque.2024.h5ad")
mard.write_h5ad("marmoset.2024.h5ad")
moud.write_h5ad("mouse.2024.h5ad")
humd.write_h5ad("human.2024.h5ad")